In [20]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import efficientnet
from tensorflow.keras.preprocessing import image
import numpy as np
import os
import shutil

In [25]:
# 1. Cargar el modelo entrenado
print("📦 Cargando modelo...")
modelo = load_model('./Estudios/Modelo/mejor_modelo_efficientnetb3_combinacion1_ft.keras')
print("✅ Modelo cargado correctamente")
print(f"Input shape: {modelo.input_shape}")
print(f"Output shape: {modelo.output_shape}")

📦 Cargando modelo...
✅ Modelo cargado correctamente
Input shape: (None, 300, 300, 3)
Output shape: (None, 11)


In [26]:
# 2. Obtener las clases desde tu dataset (opcional)
#    Cambia esta ruta a la de tu dataset de validación/entrenamiento
ruta_dataset = './Training'

if os.path.exists(ruta_dataset):
    clases = sorted([d for d in os.listdir(ruta_dataset) 
                    if os.path.isdir(os.path.join(ruta_dataset, d))])
    print(f"\n📋 Clases encontradas: {clases}")
else:
    print("\n⚠️ No se encontró el dataset. Usando nombres genéricos.")
    clases = [f"Clase_{i}" for i in range(12)]



📋 Clases encontradas: ['drinking', 'hair_and_makeup', 'phonecall_left', 'phonecall_right', 'radio', 'reach_backseat', 'reach_side', 'safe_drive', 'talking_to_passenger', 'texting_left', 'texting_right']


In [27]:
# 3. Función para predecir (usando el mismo preprocesado del entrenamiento)
def predecir_imagen(ruta_imagen, modelo, clases):
    # Cargar imagen y redimensionar a 300x300
    img = image.load_img(ruta_imagen, target_size=(300, 300))
    img_array = image.img_to_array(img)
    
    # ⭐ IMPORTANTE: Usar el mismo preprocess_input del entrenamiento ⭐
    # Esto es CLAVE para que funcione correctamente
    img_array = efficientnet.preprocess_input(img_array)
    
    # Añadir dimensión de batch
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predecir
    predicciones = modelo.predict(img_array, verbose=0)
    clase_predicha = np.argmax(predicciones[0])
    confianza = np.max(predicciones[0])
    
    return clases[clase_predicha], confianza, predicciones[0]


In [29]:
# 4. Probar con una imagen
print("\n" + "="*60)
print("🔍 PRUEBA DEL MODELO")
print("="*60)

ruta_imagen_prueba = input("\n📸 Introduce la ruta de una imagen para probar: ")

if os.path.exists(ruta_imagen_prueba):
    clase, confianza, todas_prob = predecir_imagen(ruta_imagen_prueba, modelo, clases)
    
    print(f"\n🎯 RESULTADO:")
    print(f"   📌 Clase predicha: {clase}")
    print(f"   📊 Confianza: {confianza:.2%}")
    
    # Mostrar top 3 predicciones
    print(f"\n🏆 Top 3 predicciones:")
    top_3 = np.argsort(todas_prob)[-3:][::-1]
    for i, idx in enumerate(top_3):
        print(f"   {i+1}. {clases[idx]}: {todas_prob[idx]:.2%}")
    
    # Mostrar si la confianza es buena
    if confianza > 0.9:
        print("\n✅ Excelente: El modelo está muy seguro de su predicción")
    elif confianza > 0.7:
        print("\n👍 Bueno: El modelo está razonablemente seguro")
    else:
        print("\n⚠️ Precaución: El modelo no está muy seguro. Podría ser otra clase")
        
else:
    print(f"❌ No se encontró la imagen en: {ruta_imagen_prueba}")


🔍 PRUEBA DEL MODELO



📸 Introduce la ruta de una imagen para probar:  ./Testing/reach_backseat/cuerpo_gE_26_s3_2019-03-15T09;38;23+01;00_1398.png



🎯 RESULTADO:
   📌 Clase predicha: reach_side
   📊 Confianza: 86.95%

🏆 Top 3 predicciones:
   1. reach_side: 86.95%
   2. reach_backseat: 13.05%
   3. safe_drive: 0.00%

👍 Bueno: El modelo está razonablemente seguro


In [32]:
# 5. Opcional: Probar múltiples imágenes de una carpeta
print("\n" + "="*60)
respuesta = input("\n¿Quieres probar con todas las imágenes de una carpeta? (s/n): ")

if respuesta.lower() == 's':
    carpeta_prueba = input("📁 Ruta de la carpeta con imágenes: ")
    if os.path.exists(carpeta_prueba):
        imagenes = [f for f in os.listdir(carpeta_prueba) 
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        
        if not imagenes:
            print("❌ No se encontraron imágenes en la carpeta")
        else:
            print(f"\n🖼️ Probando {min(10, len(imagenes))} imágenes...\n")
            aciertos = 0
            for img_file in imagenes[:10]:
                img_path = os.path.join(carpeta_prueba, img_file)
                try:
                    clase, confianza, _ = predecir_imagen(img_path, modelo, clases)
                    # Mostrar resultado
                    simbolo = "✅" if confianza > 0.7 else "⚠️"
                    print(f"   {simbolo} {img_file[:35]:35s} → {clase:15s} ({confianza:.1%})")
                except Exception as e:
                    print(f"   ❌ Error con {img_file}: {e}")
    else:
        print("❌ Carpeta no encontrada")


¿Quieres probar con todas las imágenes de una carpeta? (s/n):  s
📁 Ruta de la carpeta con imágenes:  ./Testing/reach_backseat



🖼️ Probando 10 imágenes...

   ✅ cuerpo_gZ_33_s3_2019-04-04T15;06;45 → reach_backseat  (96.4%)
   ✅ cara_gZ_33_s3_2019-04-04T15;06;45+0 → safe_drive      (74.4%)
   ⚠️ cuerpo_gZ_33_s3_2019-04-04T15;06;45 → reach_backseat  (63.2%)
   ⚠️ cara_gZ_33_s3_2019-04-04T15;06;45+0 → safe_drive      (42.9%)
   ⚠️ cuerpo_gZ_33_s3_2019-04-04T15;06;45 → reach_backseat  (47.7%)
   ⚠️ cara_gZ_33_s3_2019-04-04T15;06;45+0 → safe_drive      (60.7%)
   ⚠️ cuerpo_gZ_33_s3_2019-04-04T15;06;45 → reach_backseat  (56.7%)
   ⚠️ cara_gZ_33_s3_2019-04-04T15;06;45+0 → reach_backseat  (56.6%)
   ⚠️ cuerpo_gZ_33_s3_2019-04-04T15;06;45 → reach_backseat  (51.6%)
   ⚠️ cara_gZ_33_s3_2019-04-04T15;06;45+0 → reach_backseat  (67.1%)


In [40]:
# Juntamos dos clases
# Ruta a tu dataset de entrenamiento
train_path = './Testing'

# Mover imágenes de change_gear a safe_drive
carpeta_origen = os.path.join(train_path, 'phonecall_left')
carpeta_destino = os.path.join(train_path, 'phonecall')

os.makedirs(carpeta_destino, exist_ok=True)

if os.path.exists(carpeta_origen):
    for img in os.listdir(carpeta_origen):
        shutil.move(
            os.path.join(carpeta_origen, img),
            os.path.join(carpeta_destino, img)
        )
    os.rmdir(carpeta_origen)  # Eliminar carpeta vacía
    print("✅ Clases unificadas: phonecall_left → phonecall")

✅ Clases unificadas: phonecall_left → phonecall
